In [ ]:
import itertools as it

import boto3
import botocore
from cliffs_delta import cliffs_delta
from matplotlib import pyplot as plt
import matplotlib.lines as mlines
import numpy as np
import pandas as pd
from pandas.util import hash_pandas_object
import pingouin as pg
from scipy import stats as scipy_stats
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
import statsmodels.genmod.cov_struct as cov_structs
from teeplot import teeplot as tp


In [ ]:
from dishpylib.pyhelpers import make_outattr_metadata
from dishpylib.pyhelpers import print_runtime


In [ ]:
print_runtime()


In [ ]:
teeplot_subdir = "2026-07-06-pcomplexity-gee"


# get data


In [ ]:
s3_handle = boto3.resource(
    's3',
    region_name="us-east-2",
    config=botocore.config.Config(
        signature_version=botocore.UNSIGNED,
    ),
)
bucket_handle = s3_handle.Bucket('prq49')

series_profiles, = bucket_handle.objects.filter(
    Prefix=f'endeavor=16/series-profiles/stage=8+what=elaborated/',
)


In [ ]:
df = pd.read_csv(
    f's3://prq49/{series_profiles.key}',
    compression='xz',
)
dfdigest = '{:x}'.format( hash_pandas_object( df ).sum() )
dfdigest


In [ ]:
def make_outattr_metadata():
    return {}


In [ ]:
for stint in df['Stint'].unique():
    exec(f'df{stint} = df[ df["Stint"] == {stint} ]')


In [ ]:
df["m10"] = df["Stint"] % 10
dfm10 = df[ df["m10"].isin([0, 9]) ]


In [ ]:
s3_handle = boto3.resource(
    "s3",
    region_name="us-east-2",
    config=botocore.config.Config(
        signature_version=botocore.UNSIGNED,
    ),
)
bucket_handle = s3_handle.Bucket("prq49")

dfnone = pd.concat(
    [
        pd.read_csv(f"s3://prq49/{item.key}")
        for item in bucket_handle.objects.filter(
            Prefix=f"endeavor=16/external-competitions/stage=2+what=collated/",
        )
    ],
    ignore_index=True,
)
dfnone["kind"] = "none"
print(dfnone["Competition Stint"].unique())
print(dfnone["Competition Series"].unique())


In [ ]:
s3_handle = boto3.resource(
    "s3",
    region_name="us-east-2",
    config=botocore.config.Config(
        signature_version=botocore.UNSIGNED,
    ),
)
bucket_handle = s3_handle.Bucket("prq49")

dfbio = pd.concat(
    [
        pd.read_csv(f"s3://prq49/{item.key}")
        for item in bucket_handle.objects.filter(
            Prefix=f"endeavor=16/external-competitions-focalbb-true/stage=2+what=collated/",
        )
    ],
    ignore_index=True,
)
dfbio["kind"] = "eco"
print(dfbio["Competition Stint"].unique())
print(dfbio["Competition Series"].unique())


In [ ]:
s3_handle = boto3.resource(
    "s3",
    region_name="us-east-2",
    config=botocore.config.Config(
        signature_version=botocore.UNSIGNED,
    ),
)
bucket_handle = s3_handle.Bucket("prq49")

dfabio = pd.concat(
    [
        pd.read_csv(f"s3://prq49/{item.key}")
        for item in bucket_handle.objects.filter(
            Prefix=f"endeavor=16/external-competitions-focalbb/stage=2+what=collated/",
        )
    ],
    ignore_index=True,
)
dfabio["kind"] = "self"
print(dfabio["Competition Stint"].unique())
print(dfabio["Competition Series"].unique())


In [ ]:
dfx = pd.concat([dfabio, dfbio, dfnone], ignore_index=True)
dfx["Fitness Differential Focal Sign"] = np.sign(
    dfx["Fitness Differential Focal"],
)
dfx["Series"] = dfx["Competition Series"]
dfx["Stint"] = dfx["Competition Stint"]
dfx["flip series"] = dfx["genome series"] * (dfx["Root ID"] == 1)
dfx["Competitor"] = (
    dfx
    .groupby(["Stint", "Series", "kind", "Competition Repro"])
    ["flip series"]
    .transform("max")
)
dfx = dfx[
    dfx["Root ID"] == 0
].groupby(["Stint", "Series", "kind", "Competitor"])[
    "Focal Prevalence"
].mean().round().reset_index(drop=False)
dfx


In [ ]:
dfj = dfx.join(
    dfm10[
        [
            "Stint",
            "Series",
            "Fitness Complexity",
            "Flagged Advantageous Sites",
            "Flagged Deleterious Sites",
            "Cardinal Interface Complexity",
            "Cell Interface Complexity",
        ]
    ].set_index(
        ["Stint", "Series"]
    ),
    on=["Stint", "Series"],
    how="inner",
).reset_index(drop=False)
dfj["Favored"] = dfj["Focal Prevalence"] > 0.5
dfj


In [ ]:
lpal = {
    "co-evo": "#a3cf73",
    "self": "#5caf96",
    "none": "#f2dfae",
    "Bkgd": "#ffffff",
}


In [ ]:
for other in ["eco", "none"]:
    self = "self"
    # 1. Group and prepare the data
    plot_data = dfj[
        dfj["kind"].isin([self, other])
    ].replace(
        {"eco": "co-evo"},
    ).groupby(
        ["Series", "kind", "Stint"]
    ).mean().copy().reset_index()

    # 2. Calculate Wilcoxon metrics using Pingouin
    p_values_greater = {}
    p_values_two_sided = {}
    effect_sizes = {}  # Will hold Rank-Biserial Correlation (RBC)

    if other == "eco":
        other = "co-evo"

    for series in plot_data["Series"].unique():
        series_data = plot_data[plot_data["Series"] == series]

        # Align by "Stint" for exact pairs
        self_df = series_data[series_data["kind"] == self].set_index("Stint")["Focal Prevalence"]
        other_df = series_data[series_data["kind"] == other].set_index("Stint")["Focal Prevalence"]

        aligned = pd.concat([self_df, other_df], axis=1, join="inner")
        aligned.columns = ["self", "other"]

        self_vals = aligned["self"]
        other_vals = aligned["other"]

        if len(self_vals) > 0:
            # Pingouin's wilcoxon returns a DataFrame with stats, p-val, and RBC

            # One-sided test (for sorting)
            res_greater = pg.wilcoxon(self_vals, other_vals, alternative="greater", method="approx")
            p_values_greater[series] = res_greater["p-val"].values[0]

            # Two-sided test (for the column and effect size)
            res_two_sided = pg.wilcoxon(self_vals, other_vals, alternative="two-sided", method="approx")
            p_values_two_sided[series] = res_two_sided["p-val"].values[0]

            # RBC (Rank-Biserial Correlation) is the standard paired effect size
            effect_sizes[series] = res_two_sided["RBC"].values[0]

        else:
            p_values_greater[series] = float('inf')
            p_values_two_sided[series] = float('inf')
            effect_sizes[series] = float('nan')

    # Add the metrics as new columns to the dataframe
    plot_data["Two-Sided P-Value"] = plot_data["Series"].map(p_values_two_sided)
    plot_data["Rank-Biserial Effect"] = plot_data["Series"].map(effect_sizes)
    # plot_data["CLES"] = plot_data["Series"].map(effect_sizes)

    # 3. Sort Series by one-sided p-value (Ascending)
    sorted_series = sorted(p_values_greater.keys(), key=p_values_greater.get)

    # map with " ", *, **, ***
    plot_data["signif"] = plot_data["Two-Sided P-Value"].apply(lambda p: " " if p > 0.05 else "***" if p <= 0.001 else "**" if p <= 0.01 else "*")

    plot_data["order"] = plot_data["Series"].apply(sorted_series.index)

    with tp.teed(
        plt.subplots,
        2,
        1,
        constrained_layout=True,
        figsize=(8, 1.3),
        gridspec_kw={'height_ratios': [0.7, 0.6], "hspace": 0},
        sharey=False,
        teeplot_outattrs={
            "other": other,
        },
        teeplot_subdir=teeplot_subdir,
    ) as (fig, (ax1, ax2)):

        # 4. Generate the plot
        sns.violinplot(
            ax=ax2,
            data=plot_data,
            hue="kind",
            x="Series",
            y="Focal Prevalence",
            split=True,
            inner="quartile",
            cut=0,
            order=sorted_series,
            width=1.35,
            palette=lpal,
            gap=0.1,
            linewidth=0,
            common_norm=True,
            legend=True,
        )

        # Bring back the inner quartile lines by setting their width/color manually
        # Seaborn draws the inner lines as a LineCollection inside the axes
        for collection in ax2.collections:
            # This ensures the inner lines are visible even if linewidth=0 was set globally
            if hasattr(collection, 'set_linewidth'):
                # For modern seaborn, inner lines are often separate paths or lines
                pass

        ax2.set(ylim=(0, 1), ylabel="Fit", xticklabels=[], xlabel="Replicate")
        ax2.set(xlim=(-0.7, None))

        sns.barplot(
            ax=ax1,
            x="order",
            # y="CLES",
            y="Rank-Biserial Effect",
            data=plot_data,
            # marker="o",
            hue="signif",
            palette={
                " ": "gainsboro",
                "*": "darkgray",
                "**": "dimgray",
                "***": "black",
            },
            legend=False,
            gap=0.4,
        )
        ax1.axhline(0, color='gray', linestyle='--', linewidth=1)
        ax1.set_ylim(-1, 1)
        ax1.set_ylabel("   Eff\n    Size")
        ax1.set_xticklabels([])
        ax2.set_xticklabels([])
        ax1.set_xticks([])
        ax1.set_xlabel("")
        ax1.set_yticks([0, 1])
        sns.despine(ax=ax1, left=True, bottom=True)
        sns.despine(ax=ax2)

        unique_bars = plot_data[['order', 'Rank-Biserial Effect', 'signif']].drop_duplicates()
        for _, row in unique_bars.iterrows():
            y_val = -0.04
            ax1.text(
                row['order'] + 0.12,
                y_val,
                row['signif'],
                ha='center',
                va='top',
                color='black',
                clip_on=False,
                rotation=90,
                zorder=10,
                fontsize=7,
                horizontalalignment='right',
                verticalalignment='center',
            )

        handles, labels = ax2.get_legend_handles_labels()
        empty_handle = mlines.Line2D([], [], color="none")
        handles.insert(0, empty_handle)
        labels.insert(0, "Bkgd")
        ax2.legend(handles, labels)

        sns.move_legend(
            ax2,
            loc="upper right",
            bbox_to_anchor=(1, -0.1),
            ncol=3,
            columnspacing=0.8,
            title=None,
            frameon=False,
            handletextpad=0.4,
        )
        fig.tight_layout()


# Base


In [ ]:
dfjx = dfj[
    dfj["Stint"].isin(range(9, 101, 10))
].groupby(["Series", "kind", "Stint"]).mean().reset_index().groupby(
    ["Series", "kind"],
).mean().reset_index(
    drop=False,
)

fas = "Cardinal Interface Complexity"

for other in ["eco", "none"]:
    d = dfjx[
        dfjx["kind"].isin(["self", other])
    ].copy().replace({"eco": "co-evo"}).copy()
    if other == "eco":
        other = "co-evo"
    d["kind"] = pd.Categorical(
        d["kind"],
        categories=[other, "self"],
    )

    print("n Series:", d["Series"].nunique(), " n rows:", len(d))
    print(d["kind"].unique())

    gee = smf.gee(
        "Q('Focal Prevalence') ~ C(kind) * Q('Cardinal Interface Complexity')",
        groups="Series",
        data=d,
        family=sm.families.Binomial(),
        # time="Stint",
        # cov_struct=cov_structs.Autoregressive(
        #     grid=True,
        # ),
    ).fit()
    print(gee.summary())
    print(gee.cov_struct.summary())  # within-Series correlation

    cic = "Cardinal Interface Complexity"
    light_palette = {"self": "#6dceb1", "co-evo": "#a3cf73", "none": "#f2dfae"}
    dark_palette = {"self": "#137177", "co-evo": "#66a044", "none": "#dac796"}

    grid = np.linspace(d[cic].min(), d[cic].max(), 100)

    with tp.teed(
        plt.subplots,
        figsize=(3, 2),
        teeplot_outattrs={"what": "base", "other": other},
        teeplot_subdir=teeplot_subdir,
    ) as (fig, ax):
        for kind in [other, "self"]:
            newdata = pd.DataFrame({"kind": kind, cic: grid})
            pred = gee.get_prediction(newdata)
            mean = pred.predicted_mean
            ci = pred.conf_int()

            ax.plot(grid, mean, color=dark_palette[kind], lw=1, label=kind)
            ax.fill_between(
                grid, ci[:, 0], ci[:, 1], color=light_palette[kind], alpha=0.3
            )

            sub = d[d["kind"] == kind]
            ax.scatter(
                sub[cic],
                sub["Focal Prevalence"],
                color=dark_palette[kind],
                s=4,
                alpha=0.5,
                edgecolor=dark_palette[kind],
            )

        ax.set_xlabel("Phenotype Complexity")
        ax.set_ylabel("Fitness")
        ax.set_ylim(-0.02, 1.02)
        ax.set_xlim(0, None)
        ax.legend(frameon=False, title="Background", fontsize=8)
        fig.tight_layout()
        sns.despine()

        handles, labels = ax.get_legend_handles_labels()
        empty_handle = mlines.Line2D([], [], color="none")
        handles.insert(0, empty_handle)
        labels.insert(0, "Bkgd")
        ax.legend(handles, labels)
        sns.move_legend(
            ax,
            loc="upper right",
            bbox_to_anchor=(1, 0.25),
            ncol=3,
            columnspacing=0.8,
            title=None,
            frameon=False,
            handletextpad=0.4,
            handlelength=1.0,
        )


# Exclude 16005


In [ ]:
dfjx = dfj[
    dfj["Stint"].isin(range(9, 101, 10))
    & (dfj["Series"] != 16005)
].groupby(["Series", "kind", "Stint"]).mean().reset_index().groupby(
    ["Series", "kind"],
).mean().reset_index(
    drop=False,
)

fas = "Cardinal Interface Complexity"

for other in ["eco", "none"]:
    d = dfjx[
        dfjx["kind"].isin(["self", other])
    ].copy().replace({"eco": "co-evo"}).copy()
    if other == "eco":
        other = "co-evo"
    d["kind"] = pd.Categorical(
        d["kind"],
        categories=[other, "self"],
    )

    print("n Series:", d["Series"].nunique(), " n rows:", len(d))

    gee = smf.gee(
        "Q('Focal Prevalence') ~ C(kind) * Q('Cardinal Interface Complexity')",
        groups="Series",
        data=d,
        family=sm.families.Binomial(),
        # time="Stint",
        # cov_struct=cov_structs.Autoregressive(
        #     grid=True,
        # ),
    ).fit()
    print(gee.summary())
    print(gee.cov_struct.summary())  # within-Series correlation

    cic = "Cardinal Interface Complexity"
    light_palette = {"self": "#6dceb1", "co-evo": "#a3cf73", "none": "#f2dfae"}
    dark_palette = {"self": "#137177", "co-evo": "#66a044", "none": "#dac796"}

    grid = np.linspace(d[cic].min(), d[cic].max(), 100)

    with tp.teed(
        plt.subplots,
        figsize=(3, 2),
        teeplot_outattrs={"what": "exclude", "other": other},
        teeplot_subdir=teeplot_subdir,
    ) as (fig, ax):
        for kind in [other, "self"]:
            newdata = pd.DataFrame({"kind": kind, cic: grid})
            pred = gee.get_prediction(newdata)
            mean = pred.predicted_mean
            ci = pred.conf_int()

            ax.plot(grid, mean, color=dark_palette[kind], lw=1, label=kind)
            ax.fill_between(
                grid, ci[:, 0], ci[:, 1], color=light_palette[kind], alpha=0.3
            )

            sub = d[d["kind"] == kind]
            ax.scatter(
                sub[cic],
                sub["Focal Prevalence"],
                color=dark_palette[kind],
                s=4,
                alpha=0.5,
                edgecolor=dark_palette[kind],
                zorder=10,
            )

        ax.set_xlabel("Phenotype Complexity")
        ax.set_ylabel("Fitness")
        ax.set_ylim(-0.02, 1.02)
        ax.set_xlim(0, None)
        ax.legend(frameon=False, title="Background", fontsize=8)
        fig.tight_layout()
        sns.despine()

        handles, labels = ax.get_legend_handles_labels()
        empty_handle = mlines.Line2D([], [], color="none")
        handles.insert(0, empty_handle)
        labels.insert(0, "Bkgd")
        ax.legend(handles, labels)
        sns.move_legend(
            ax,
            loc="upper right",
            bbox_to_anchor=(1, 0.25),
            ncol=3,
            columnspacing=0.8,
            title=None,
            frameon=False,
            handletextpad=0.4,
            handlelength=1.0,
        )


# Disaggregate


In [ ]:
dfjx = dfj[
    dfj["Stint"].isin(range(9, 101, 10))
].groupby(["Series", "kind", "Stint"]).mean().reset_index(
    drop=False,
)

fas = "Cardinal Interface Complexity"

for other in ["eco", "none"]:
    d = dfjx[
        dfjx["kind"].isin(["self", other])
    ].copy().replace({"eco": "co-evo"}).copy()
    if other == "eco":
        other = "co-evo"
    d["kind"] = pd.Categorical(
        d["kind"],
        categories=[other, "self"],
    )

    print("n Series:", d["Series"].nunique(), " n rows:", len(d))
    print(d["kind"].unique())

    gee = smf.gee(
        "Q('Focal Prevalence') ~ C(kind) * Q('Cardinal Interface Complexity')",
        groups="Series",
        data=d,
        family=sm.families.Binomial(),
        time="Stint",
        cov_struct=cov_structs.Autoregressive(
            grid=True,
        ),
    ).fit()
    print(gee.summary())
    print(gee.cov_struct.summary())  # within-Series correlation

    cic = "Cardinal Interface Complexity"
    light_palette = {"self": "#6dceb1", "co-evo": "#a3cf73", "none": "#f2dfae"}
    dark_palette = {"self": "#137177", "co-evo": "#66a044", "none": "#dac796"}

    grid = np.linspace(d[cic].min(), d[cic].max(), 100)

    with tp.teed(
        plt.subplots,
        figsize=(3, 2),
        teeplot_outattrs={"what": "disagg", "other": other},
        teeplot_subdir=teeplot_subdir,
    ) as (fig, ax):
        for kind in [other, "self"]:
            newdata = pd.DataFrame({"kind": kind, cic: grid})
            pred = gee.get_prediction(newdata)
            mean = pred.predicted_mean
            ci = pred.conf_int()

            ax.plot(grid, mean, color=dark_palette[kind], lw=1, label=kind)
            ax.fill_between(
                grid, ci[:, 0], ci[:, 1], color=light_palette[kind], alpha=0.3
            )

            sub = d[d["kind"] == kind]
            ax.scatter(
                sub[cic],
                sub["Focal Prevalence"],
                color=dark_palette[kind],
                s=4,
                alpha=0.5,
                edgecolor=dark_palette[kind],
            )

        ax.set_xlabel("Phenotype Complexity")
        ax.set_ylabel("Fitness")
        ax.set_ylim(-0.02, 1.02)
        ax.set_xlim(-1, None)
        ax.legend(frameon=False, title="Background", fontsize=8)
        fig.tight_layout()
        sns.despine()

        handles, labels = ax.get_legend_handles_labels()
        empty_handle = mlines.Line2D([], [], color="none")
        handles.insert(0, empty_handle)
        labels.insert(0, "Bkgd")
        ax.legend(handles, labels)
        sns.move_legend(
            ax,
            loc="lower center",
            bbox_to_anchor=(0.5, 1),
            ncol=3,
            columnspacing=0.8,
            title=None,
            frameon=False,
            handletextpad=0.4,
            handlelength=1.0,
        )


# Disagg/Exclude


In [ ]:
dfjx = dfj[
    dfj["Stint"].isin(range(9, 101, 10))
    & (dfj["Series"] != 16005)
].groupby(["Series", "kind", "Stint"]).mean().reset_index(
    drop=False,
)

fas = "Cardinal Interface Complexity"

for other in ["eco", "none"]:
    d = dfjx[
        dfjx["kind"].isin(["self", other])
    ].copy().replace({"eco": "co-evo"}).copy()
    if other == "eco":
        other = "co-evo"
    d["kind"] = pd.Categorical(
        d["kind"],
        categories=[other, "self"],
    )

    print("n Series:", d["Series"].nunique(), " n rows:", len(d))
    print(d["kind"].unique())

    gee = smf.gee(
        "Q('Focal Prevalence') ~ C(kind) * Q('Cardinal Interface Complexity')",
        groups="Series",
        data=d,
        family=sm.families.Binomial(),
        time="Stint",
        cov_struct=cov_structs.Autoregressive(
            grid=True,
        ),
    ).fit()
    print(gee.summary())
    print(gee.cov_struct.summary())  # within-Series correlation

    cic = "Cardinal Interface Complexity"
    light_palette = {"self": "#6dceb1", "co-evo": "#a3cf73", "none": "#f2dfae"}
    dark_palette = {"self": "#137177", "co-evo": "#66a044", "none": "#dac796"}

    grid = np.linspace(d[cic].min(), d[cic].max(), 100)

    with tp.teed(
        plt.subplots,
        figsize=(3, 2),
        teeplot_outattrs={"what": "disagg-excl", "other": other},
        teeplot_subdir=teeplot_subdir,
    ) as (fig, ax):
        for kind in [other, "self"]:
            newdata = pd.DataFrame({"kind": kind, cic: grid})
            pred = gee.get_prediction(newdata)
            mean = pred.predicted_mean
            ci = pred.conf_int()

            ax.plot(grid, mean, color=dark_palette[kind], lw=1, label=kind)
            ax.fill_between(
                grid, ci[:, 0], ci[:, 1], color=light_palette[kind], alpha=0.3
            )

            sub = d[d["kind"] == kind]
            ax.scatter(
                sub[cic],
                sub["Focal Prevalence"],
                color=dark_palette[kind],
                s=4,
                alpha=0.5,
                edgecolor=dark_palette[kind],
            )

        ax.set_xlabel("Phenotype Complexity")
        ax.set_ylabel("Fitness")
        ax.set_ylim(-0.02, 1.02)
        ax.set_xlim(-1, None)
        ax.legend(frameon=False, title="Background", fontsize=8)
        fig.tight_layout()
        sns.despine()

        handles, labels = ax.get_legend_handles_labels()
        empty_handle = mlines.Line2D([], [], color="none")
        handles.insert(0, empty_handle)
        labels.insert(0, "Bkgd")
        ax.legend(handles, labels)
        sns.move_legend(
            ax,
            loc="lower center",
            bbox_to_anchor=(0.5, 1),
            ncol=3,
            columnspacing=0.8,
            title=None,
            frameon=False,
            handletextpad=0.4,
            handlelength=1.0,
        )
